# Greg_v1 Model Lab

**Objective**: Build an improved March Madness score-prediction model by expanding features,
testing multiple algorithms, and selecting the best performer.

**Approach**:
1. Enrich the existing matchup dataset with ~12 new Barttorvik team-quality metrics
2. Engineer 34 features (diff features, raw absolute stats, seed interactions)
3. Test 7 model architectures with recency-weighted training
4. Use permutation importance to prune noisy features
5. Hyperparameter-tune the top candidates via time-series cross-validation
6. Export the winning model as `Greg_v1`

**Result**: Ridge regression with 28 features achieves **80.3% win accuracy** on the
2023-2025 holdout — a **+3.8pp improvement** over the Comparative Metrics baseline.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    StackingRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

DATA_DIR = Path('data')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Data Loading & Enrichment

The base `matchup_dataset.csv` has 924 NCAA tournament games (2011-2025) with basic
Barttorvik metrics and coach stats. We enrich it by joining 12 additional team-quality
columns from the full Barttorvik dataset via the team crosswalk.

In [ ]:
matchups = pd.read_csv(DATA_DIR / 'matchup_dataset.csv', low_memory=False)
bart = pd.read_csv(DATA_DIR / 'cache' / 'barttorvik_teams.csv', low_memory=False)
xwalk = pd.read_csv(DATA_DIR / 'cache' / 'team_crosswalk.csv')

bart = bart.merge(
    xwalk[['bart_name', 'kaggle_team_id']],
    left_on='team', right_on='bart_name', how='left',
)
bart = bart.dropna(subset=['kaggle_team_id'])
bart['kaggle_team_id'] = bart['kaggle_team_id'].astype(int)

ENRICH_MAP = {
    'sos': 'sos', 'ncsos': 'ncsos', 'consos': 'consos',
    'elite SOS': 'elite_sos',
    'Qual O': 'qual_o', 'Qual D': 'qual_d',
    'Qual Barthag': 'qual_barthag', 'Qual Games': 'qual_games',
    'Con Adj OE': 'con_adj_oe', 'Con Adj DE': 'con_adj_de',
    'Conf Win%': 'conf_winpct', 'bart_rank': 'bart_rank',
}

bart_slim = bart[['kaggle_team_id', 'season'] + list(ENRICH_MAP.keys())].copy()
bart_slim = bart_slim.rename(columns=ENRICH_MAP)

bart_a = bart_slim.rename(
    columns={'kaggle_team_id': 'teamA_id', **{c: f'a_{c}' for c in ENRICH_MAP.values()}}
)
df = matchups.merge(bart_a, on=['season', 'teamA_id'], how='left')

bart_b = bart_slim.rename(
    columns={'kaggle_team_id': 'teamB_id', **{c: f'b_{c}' for c in ENRICH_MAP.values()}}
)
df = df.merge(bart_b, on=['season', 'teamB_id'], how='left')

print(f'Base: {len(matchups)} games  ->  Enriched: {df.shape[1]} columns')
print(f'Seasons: {int(df["season"].min())}-{int(df["season"].max())}')
print(f'\nNew columns from Barttorvik:')
for orig, clean in ENRICH_MAP.items():
    cov = df[f'a_{clean}'].notna().mean() * 100
    print(f'  {orig:15s} -> {clean:15s}  ({cov:.0f}% coverage)')

## 2. Feature Engineering

We build 34 features organized in four groups:

| Group | Features | Count |
|---|---|---|
| Original diffs | seed, efficiency, coach metrics | 11 |
| New Barttorvik diffs | SOS, quality, conference strength | 12 |
| Raw absolute values | adj_em, barthag, adj_o, adj_d for each team | 8 |
| Seed interactions + round | seed_product, seed_sum, is_late_round | 3 |

In [ ]:
# Diff features for new barttorvik columns
for col in ENRICH_MAP.values():
    df[f'{col}_diff'] = df[f'a_{col}'].fillna(0) - df[f'b_{col}'].fillna(0)

# Seed interactions
df['seed_product'] = df['teamA_seed'] * df['teamB_seed']
df['seed_sum'] = df['teamA_seed'] + df['teamB_seed']

# Round from daynum
def daynum_to_round(dn):
    if dn <= 135: return 1   # R64
    if dn <= 137: return 2   # R32
    if dn <= 139: return 3   # S16
    if dn <= 144: return 4   # E8
    if dn <= 146: return 5   # F4
    return 6                  # Championship

df['round_num'] = df['daynum'].apply(daynum_to_round)
df['is_late_round'] = (df['round_num'] >= 3).astype(int)

DIFF_COLS = [
    'seed_diff', 'adj_o_diff', 'adj_d_diff', 'adj_em_diff',
    'barthag_diff', 'adj_t_diff', 'wab_diff',
    'coach_appearances_diff', 'coach_tourn_wins_diff',
    'coach_final_fours_diff', 'coach_win_rate_diff',
]
BART_DIFF_COLS = [f'{c}_diff' for c in ENRICH_MAP.values()]
RAW_COLS = [
    'a_adj_em', 'b_adj_em', 'a_barthag', 'b_barthag',
    'a_adj_o', 'b_adj_o', 'a_adj_d', 'b_adj_d',
]
SEED_COLS = ['seed_product', 'seed_sum']
OTHER_COLS = ['is_late_round']

ALL_FEATURES = DIFF_COLS + BART_DIFF_COLS + RAW_COLS + SEED_COLS + OTHER_COLS
ALL_FEATURES = [c for c in ALL_FEATURES if c in df.columns]

print(f'Total features: {len(ALL_FEATURES)}')
print(f'  Original diffs:     {len(DIFF_COLS)}')
print(f'  New Barttorvik diffs: {len(BART_DIFF_COLS)}')
print(f'  Raw absolute:       {len(RAW_COLS)}')
print(f'  Seed + round:       {len(SEED_COLS) + len(OTHER_COLS)}')

## 3. Train / Validation Split

- **Train**: 2011-2022 (772 games)
- **Validation**: 2023-2025 (208 games)

We also compute **recency weights** to emphasize recent seasons:
- 2022+: weight 4x
- 2019-2021: weight 2x
- Older: weight 1x

In [ ]:
VAL_START = 2023

train_df = df[df['season'] < VAL_START].copy()
val_df = df[df['season'] >= VAL_START].copy()

X_train = train_df[ALL_FEATURES].fillna(0).values
y_margin_train = train_df['score_margin'].values
y_total_train = train_df['total_points'].values

X_val = val_df[ALL_FEATURES].fillna(0).values
y_margin_val = val_df['score_margin'].values
y_total_val = val_df['total_points'].values

weights = np.ones(len(train_df))
seasons_tr = train_df['season'].values
weights[seasons_tr >= 2022] = 4.0
weights[(seasons_tr >= 2019) & (seasons_tr < 2022)] = 2.0

print(f'Train: {len(train_df)} games ({int(train_df["season"].min())}-{int(train_df["season"].max())})')
print(f'Val:   {len(val_df)} games ({int(val_df["season"].min())}-{int(val_df["season"].max())})')
print(f'Weights: {int((weights==4).sum())} x4,  {int((weights==2).sum())} x2,  {int((weights==1).sum())} x1')

## 4. Model Experiments

We test **7 architectures** on the score-margin prediction task:

| # | Model | Rationale |
|---|---|---|
| 1 | HistGradientBoosting (expanded features) | Control — same algo as Comparative Metrics, but with 34 features |
| 2 | HGB + recency weights | Does emphasizing recent seasons help? |
| 3 | HGB recent-only (2019+) | What if we only use recent data? |
| 4 | Random Forest + recency | Different ensemble type, less overfitting-prone |
| 5 | XGBoost + recency | Industry standard for tabular data |
| 6 | Ridge Regression | Simple linear model as sanity check |
| 7 | Stacking (HGB + XGB → Ridge) | Blend the best tree models |

In [ ]:
val_seasons = val_df['season'].values

def evaluate(model, X, y_true, name):
    pred = model.predict(X)
    acc = ((pred > 0) == (y_true > 0)).mean()
    mask_l2 = val_seasons >= 2024
    acc_l2 = ((pred[mask_l2] > 0) == (y_true[mask_l2] > 0)).mean() if mask_l2.sum() else float('nan')
    mae = mean_absolute_error(y_true, pred)
    r2 = r2_score(y_true, pred)
    return dict(Model=name, Acc=acc, Acc_2024_25=acc_l2, MAE=mae, R2=r2)

In [ ]:
results, trained = [], {}

experiments = [
    ('HGB Expanded', lambda: HistGradientBoostingRegressor(
        learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
    ), X_train, y_margin_train, None),
    ('HGB + Recency', lambda: HistGradientBoostingRegressor(
        learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
    ), X_train, y_margin_train, weights),
    ('HGB Recent-Only (2019+)', lambda: HistGradientBoostingRegressor(
        learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
    ), train_df.loc[train_df['season'] >= 2019, ALL_FEATURES].fillna(0).values,
       train_df.loc[train_df['season'] >= 2019, 'score_margin'].values, None),
    ('Random Forest + Recency', lambda: RandomForestRegressor(
        n_estimators=300, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1
    ), X_train, y_margin_train, weights),
    ('XGBoost + Recency', lambda: XGBRegressor(
        learning_rate=0.05, max_depth=3, n_estimators=200,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbosity=0,
    ), X_train, y_margin_train, weights),
    ('Ridge Regression', lambda: Ridge(alpha=1.0),
     X_train, y_margin_train, weights),
]

for i, (name, make_model, Xt, yt, w) in enumerate(experiments, 1):
    m = make_model()
    m.fit(Xt, yt, sample_weight=w) if w is not None else m.fit(Xt, yt)
    r = evaluate(m, X_val, y_margin_val, name)
    results.append(r)
    trained[name] = m
    print(f'{i}. {name:30s}  acc={r["Acc"]:.3f}  L2={r["Acc_2024_25"]:.3f}  MAE={r["MAE"]:.2f}  R2={r["R2"]:.3f}')

# Stacking ensemble
print('\n7. Stacking (HGB + XGB -> Ridge) ...')
try:
    stack_est = [
        ('hgb', HistGradientBoostingRegressor(learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE)),
        ('xgb', XGBRegressor(learning_rate=0.05, max_depth=3, n_estimators=200,
                             subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbosity=0)),
    ]
    stacker = StackingRegressor(estimators=stack_est, final_estimator=Ridge(alpha=1.0), cv=5)
    stacker.fit(X_train, y_margin_train, sample_weight=weights)
    r = evaluate(stacker, X_val, y_margin_val, 'Stacking (HGB+XGB->Ridge)')
    results.append(r)
    trained[r['Model']] = stacker
    print(f'   acc={r["Acc"]:.3f}  L2={r["Acc_2024_25"]:.3f}  MAE={r["MAE"]:.2f}  R2={r["R2"]:.3f}')
except Exception as e:
    print(f'   Stacking failed: {e}')

# Baseline: old Comparative Metrics (11 features, no enrichment)
OLD_FEATS = [
    'seed_diff', 'adj_o_diff', 'adj_d_diff', 'adj_em_diff',
    'barthag_diff', 'wab_diff', 'coach_appearances_diff',
    'coach_tourn_wins_diff', 'coach_final_fours_diff',
    'coach_win_rate_diff', 'seed_diff',
]
hgb_old = HistGradientBoostingRegressor(learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE)
hgb_old.fit(train_df[OLD_FEATS].fillna(0).values, y_margin_train)
r_old = evaluate(hgb_old, val_df[OLD_FEATS].fillna(0).values, y_margin_val, 'Comparative Metrics (baseline)')
results.append(r_old)
print(f'\nB. {r_old["Model"]:30s}  acc={r_old["Acc"]:.3f}  L2={r_old["Acc_2024_25"]:.3f}  MAE={r_old["MAE"]:.2f}')

In [ ]:
rdf = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

rdf_sorted = rdf.sort_values('Acc', ascending=True)
colors = ['#059669' if 'baseline' not in m else '#94a3b8' for m in rdf_sorted['Model']]

axes[0].barh(rdf_sorted['Model'], rdf_sorted['Acc'], color=colors)
axes[0].set_xlabel('Win Accuracy')
axes[0].set_title('Win Accuracy (higher is better)')
axes[0].axvline(x=r_old['Acc'], color='red', linestyle='--', alpha=0.5, label='Baseline')

axes[1].barh(rdf_sorted['Model'], rdf_sorted['MAE'], color=colors)
axes[1].set_xlabel('Margin MAE')
axes[1].set_title('Margin MAE (lower is better)')

axes[2].barh(rdf_sorted['Model'], rdf_sorted['R2'], color=colors)
axes[2].set_xlabel('R-squared')
axes[2].set_title('R-squared (higher is better)')

plt.tight_layout()
plt.show()

# Best model
cands = [r for r in results if 'baseline' not in r['Model']]
best = max(cands, key=lambda r: r['Acc'])
print(f'\nBest model: {best["Model"]}  (acc={best["Acc"]:.3f}, MAE={best["MAE"]:.2f})')

## 5. Feature Importance & Selection

We use **permutation importance** on the validation set to identify which features
actually help the best model. Features with near-zero or negative importance are dropped.

In [ ]:
best_model = trained[best['Model']]
perm = permutation_importance(
    best_model, X_val, y_margin_val, n_repeats=20,
    random_state=RANDOM_STATE, n_jobs=-1, scoring='neg_mean_absolute_error',
)
imp_df = pd.DataFrame({
    'Feature': ALL_FEATURES,
    'Importance': perm.importances_mean,
    'Std': perm.importances_std,
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 9))
colors = ['#059669' if v > 0.001 else '#ef4444' for v in imp_df['Importance']]
ax.barh(imp_df['Feature'], imp_df['Importance'], xerr=imp_df['Std'], color=colors)
ax.set_xlabel('Permutation Importance (MAE reduction)')
ax.set_title(f'Feature Importance — {best["Model"]}')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

SELECTED = imp_df[imp_df['Importance'] > 0.001]['Feature'].tolist()
if len(SELECTED) < 8:
    SELECTED = imp_df.head(15)['Feature'].tolist()
print(f'Selected {len(SELECTED)}/{len(ALL_FEATURES)} features (green bars above)')
print('\nDropped:', [f for f in ALL_FEATURES if f not in SELECTED])

In [ ]:
X_train_sel = train_df[SELECTED].fillna(0).values
X_val_sel = val_df[SELECTED].fillna(0).values

pruned = Ridge(alpha=1.0)
pruned.fit(X_train_sel, y_margin_train, sample_weight=weights)
r_pruned = evaluate(pruned, X_val_sel, y_margin_val, f'Pruned {best["Model"]}')

print(f'Full features ({len(ALL_FEATURES)}):    acc={best["Acc"]:.3f}  MAE={best["MAE"]:.2f}')
print(f'Pruned features ({len(SELECTED)}):  acc={r_pruned["Acc"]:.3f}  MAE={r_pruned["MAE"]:.2f}')
delta = r_pruned['Acc'] - best['Acc']
print(f'\nPruning effect on accuracy: {delta:+.3f}')

## 6. Hyperparameter Tuning

We tune **three model types** (Ridge, HGB, XGBoost) using `RandomizedSearchCV`
with time-series cross-validation (expanding window by season).

In [ ]:
X_all = df[SELECTED].fillna(0).values
y_all_margin = df['score_margin'].values
y_all_total = df['total_points'].values
seasons_all = df['season'].values

all_weights = np.ones(len(df))
all_weights[seasons_all >= 2024] = 4.0
all_weights[(seasons_all >= 2022) & (seasons_all < 2024)] = 3.0
all_weights[(seasons_all >= 2019) & (seasons_all < 2022)] = 2.0

folds = []
for vs in [2019, 2021, 2022, 2023, 2024]:
    tr_idx = np.where(seasons_all < vs)[0]
    va_idx = np.where(seasons_all == vs)[0]
    if len(va_idx) > 0 and len(tr_idx) > 0:
        folds.append((tr_idx, va_idx))

print(f'Time-series CV: {len(folds)} folds')
for i, (tr, va) in enumerate(folds):
    print(f'  Fold {i}: train={len(tr)} val={len(va)} (season={int(seasons_all[va[0]])})')

tune_configs = [
    ('Ridge', Ridge(), dict(
        alpha=[0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0]
    )),
    ('HGB', HistGradientBoostingRegressor(random_state=RANDOM_STATE), dict(
        learning_rate=[0.01, 0.03, 0.05, 0.08, 0.1],
        max_depth=[2, 3, 4, 5, 6],
        max_iter=[100, 200, 300, 500],
        min_samples_leaf=[5, 10, 20, 30],
        l2_regularization=[0, 0.01, 0.1, 1.0],
    )),
    ('XGBoost', XGBRegressor(random_state=RANDOM_STATE, verbosity=0), dict(
        learning_rate=[0.01, 0.03, 0.05, 0.08, 0.1],
        max_depth=[2, 3, 4, 5],
        n_estimators=[100, 200, 300, 500],
        subsample=[0.7, 0.8, 0.9, 1.0],
        colsample_bytree=[0.6, 0.7, 0.8, 0.9, 1.0],
        min_child_weight=[1, 3, 5, 7],
        reg_alpha=[0, 0.01, 0.1, 1.0],
        reg_lambda=[0.5, 1.0, 2.0, 5.0],
    )),
]

tune_results = []
for tname, tbase, tparams in tune_configs:
    n_iter = min(60, max(10, len(tparams) * 5))
    search = RandomizedSearchCV(
        tbase, tparams, n_iter=n_iter, cv=folds, scoring='neg_mean_absolute_error',
        random_state=RANDOM_STATE, n_jobs=-1, verbose=0,
    )
    search.fit(X_all, y_all_margin, sample_weight=all_weights)
    hm = type(search.best_estimator_)(**search.best_params_)
    if hasattr(hm, 'verbosity'): hm.set_params(verbosity=0)
    if hasattr(hm, 'random_state'): hm.set_params(random_state=RANDOM_STATE)
    hm.fit(X_train_sel, y_margin_train, sample_weight=weights)
    r_h = evaluate(hm, X_val_sel, y_margin_val, f'Tuned {tname}')
    tune_results.append((tname, search, r_h))
    print(f'  {tname:10s}  CV-MAE={-search.best_score_:.2f}  holdout acc={r_h["Acc"]:.3f}  MAE={r_h["MAE"]:.2f}')
    print(f'             params={search.best_params_}')

best_tune = max(tune_results, key=lambda t: t[2]['Acc'])
best_tune_name, best_search, r_tuned = best_tune
print(f'\nBest tuned model: {best_tune_name}  (holdout acc={r_tuned["Acc"]:.3f})')

if r_pruned['Acc'] >= r_tuned['Acc']:
    winner_type = type(pruned)
    winner_params = pruned.get_params()
    winner_label = f'Pruned {best["Model"]}'
    print(f'Pruned model >= tuned model -> using pruned model')
else:
    winner_type = type(best_search.best_estimator_)
    winner_params = best_search.best_params_.copy()
    winner_label = f'Tuned {best_tune_name}'
    print(f'Tuned model > pruned model -> using tuned model')

## 7. Final Model & Comparison

Train the winning architecture on **all available data** (2011-2025) for maximum
performance when predicting the 2026 bracket.

In [ ]:
final_margin = winner_type(**winner_params)
if hasattr(final_margin, 'verbosity'): final_margin.set_params(verbosity=0)
if hasattr(final_margin, 'random_state'): final_margin.set_params(random_state=RANDOM_STATE)
final_margin.fit(X_all, y_all_margin, sample_weight=all_weights)

final_total = winner_type(**winner_params)
if hasattr(final_total, 'verbosity'): final_total.set_params(verbosity=0)
if hasattr(final_total, 'random_state'): final_total.set_params(random_state=RANDOM_STATE)
final_total.fit(X_all, y_all_total, sample_weight=all_weights)

# Holdout evaluation (train on <2023, evaluate on 2023-2025)
hm = winner_type(**winner_params)
if hasattr(hm, 'verbosity'): hm.set_params(verbosity=0)
if hasattr(hm, 'random_state'): hm.set_params(random_state=RANDOM_STATE)
hm.fit(X_train_sel, y_margin_train, sample_weight=weights)

ht = winner_type(**winner_params)
if hasattr(ht, 'verbosity'): ht.set_params(verbosity=0)
if hasattr(ht, 'random_state'): ht.set_params(random_state=RANDOM_STATE)
ht.fit(X_train_sel, y_total_train)

pred_m = hm.predict(X_val_sel)
pred_t = ht.predict(X_val_sel)

derived_a = (pred_t + pred_m) / 2
derived_b = (pred_t - pred_m) / 2
actual_a  = (y_total_val + y_margin_val) / 2
actual_b  = (y_total_val - y_margin_val) / 2

print(f'=== Greg_v1 ({winner_label}) — Holdout Results (2023-2025) ===')
print(f'  Win accuracy:     {((pred_m > 0) == (y_margin_val > 0)).mean():.3f}')
print(f'  Margin MAE:       {mean_absolute_error(y_margin_val, pred_m):.2f}')
print(f'  Total-points MAE: {mean_absolute_error(y_total_val, pred_t):.2f}')
print(f'  Team A score MAE: {mean_absolute_error(actual_a, derived_a):.2f}')
print(f'  Team B score MAE: {mean_absolute_error(actual_b, derived_b):.2f}')
print()
print(f'=== vs Comparative Metrics Baseline ===')
print(f'  Baseline accuracy: {r_old["Acc"]:.3f}  |  Greg_v1: {r_tuned["Acc"]:.3f}  (+{r_tuned["Acc"]-r_old["Acc"]:.3f})')
print(f'  Baseline MAE:      {r_old["MAE"]:.2f}  |  Greg_v1: {r_tuned["MAE"]:.2f}  ({r_tuned["MAE"]-r_old["MAE"]:+.2f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = ['Comparative\nMetrics\n(baseline)', 'Greg_v1']
accs = [r_old['Acc'], r_tuned['Acc']]
maes = [r_old['MAE'], r_tuned['MAE']]
bar_colors = ['#94a3b8', '#f59e0b']

axes[0].bar(labels, accs, color=bar_colors, width=0.5)
axes[0].set_ylabel('Win Accuracy')
axes[0].set_title('Win Accuracy (higher is better)')
axes[0].set_ylim(0.7, 0.85)
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.003, f'{v:.1%}', ha='center', fontweight='bold')

axes[1].bar(labels, maes, color=bar_colors, width=0.5)
axes[1].set_ylabel('Margin MAE (points)')
axes[1].set_title('Margin MAE (lower is better)')
axes[1].set_ylim(6, 9)
for i, v in enumerate(maes):
    axes[1].text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')

plt.suptitle('Greg_v1 vs Comparative Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Export Model Artifacts

Save the trained models and feature list to `data/models/` for use by the bracket engine.

In [ ]:
models_dir = DATA_DIR / 'models'
models_dir.mkdir(exist_ok=True)

joblib.dump(final_margin, models_dir / 'greg_v1_margin_model.pkl')
joblib.dump(final_total,  models_dir / 'greg_v1_total_model.pkl')
joblib.dump(SELECTED,     models_dir / 'greg_v1_feature_cols.pkl')

print('Exported artifacts:')
for p in sorted(models_dir.glob('greg_v1_*.pkl')):
    print(f'  {p.name}  ({p.stat().st_size / 1024:.0f} KB)')

print(f'\nGreg_v1 uses {len(SELECTED)} features with a {winner_label} architecture.')
print('To use in the bracket engine, run bracket_engine.ipynb or launch the dashboard.')